# MAYBE DELETE NOTEBOOK IF EVAL MODE FROM CLI WORKS IN YAIB SETUP

In [1]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var

from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *

import torch
from icu_benchmarks.models.dl_models.bat import * 


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


#### Load weights

In [ ]:
#old_ckpt_path = "/work3/s185395/yaib_logs/mimic/Mortality24/BAT/2025-05-31T11-18-01/repetition_0/fold_0/model.ckpt"
new_ckpt_path = "/work3/s185395/yaib_logs/mimic/Mortality24/BAT/2025-06-04T09-47-38/repetition_0/fold_0/model.ckpt"

# Load raw checkpoint dicts
#old_ckpt = torch.load(old_ckpt_path, map_location="cpu")
new_ckpt = torch.load(new_ckpt_path, map_location="cpu")


/tmp/ipykernel_2830809/3610754381.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_ckpt = torch.load(old_ckpt_path, map_location="cpu")
/tmp/ipykernel_2830809/3610754

In [6]:
# Extract hyperparameters
#old_hparams = old_ckpt.get("hyper_parameters", {})
new_hparams = new_ckpt.get("hyper_parameters", {})

# Display what was saved before and now
#print("🔍 Old checkpoint hyperparameters:")
#for k, v in old_hparams.items():
#    print(f"  {k}: {v}")

print("\n✅ New checkpoint hyperparameters:")
for k, v in new_hparams.items():
    print(f"  {k}: {v}")



✅ New checkpoint hyperparameters:
  input_size: torch.Size([256, 48, 25])
  lr: 0.0001
  pooling: max
  epochs: 100
  run_mode: Classification
  cpu: False
  input_shape: None
  momentum: 0.9
  lr_scheduler: None
  lr_factor: 0.99
  lr_steps: None
  initialization_method: normal
  value_embed_size: 32
  layers: 2
  heads: 1
  dropout: 0
  attn_dropout: 0
  use_mask: False
  prediction_head: <class 'icu_benchmarks.models.dl_models.bat.BinaryClassificationHead'>
  prediction_head_kwargs: {'num_classes': 2}


In [79]:
model_path = "/work3/s185395/yaib_logs/mimic/Mortality24/BAT/2025-05-31T11-18-01/repetition_0/fold_0/model.ckpt"
ckpt = torch.load(model_path, map_location="cpu")

state_dict = {
    k.replace("model.", "", 1): v
    for k, v in ckpt["state_dict"].items()
}

filtered_state_dict = {k: v for k, v in state_dict.items() if not k.startswith("logit.")}


/tmp/ipykernel_2599137/1128744625.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")


#### Load weights into model 

In [80]:
sensors_count = 48
T = 24


model = BAT(
    input_size=(None, sensors_count, T),  # Adjust None with batch size or leave as placeholder
    value_embed_size=32,
    layers=2,
    heads=1,
    dropout=0,
    attn_dropout=0,
    use_mask=False,
    prediction_head=BinaryClassificationHead,
    prediction_head_kwargs={"num_classes": 2}
)

model.eval()  # Set model to evaluation mode
model.to("cpu")

# Dummy inputs (must match expected shape)
dummy_data = torch.randn(1, sensors_count,T)
dummy_static = torch.randn(1, 4)
dummy_time = torch.randn(1, sensors_count, T)
dummy_time = torch.arange(T).unsqueeze(0).repeat(1, 1)  
dummy_mask = torch.ones(1, sensors_count, T)

# Forward pass (this creates self.head)
with torch.no_grad():
    _ = model.model(dummy_data, dummy_static, dummy_time, dummy_mask)

model.model.load_state_dict(filtered_state_dict) 



use static


<All keys matched successfully>

#### Load data

In [81]:
vars_dict = {
    "GROUP": "stay_id",
    "SEQUENCE": "time",
    "LABEL": "label",
    "DYNAMIC": ["alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck", "ckmb", "cl",
        "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map", "mch", "mchc", "mcv",
        "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine",
        "wbc"],
    "STATIC": ["age", "sex", "height", "weight"],
}

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin")

ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [82]:
repetition_index=0
fold_index=0
cv_repetitions=5
cv_folds=5

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("/work3/s185395/YAIB-cohorts/data/mortality24/mimic"),
    #data_dir=Path("/work3/s185395/YAIB-cohorts/data/mortality24/eicu"),
    #data_dir=Path("/work3/s185395/YAIB-cohorts/data/mortality24/miiv"),
    repetition_index=repetition_index,
    fold_index=fold_index,
    cv_repetitions=cv_repetitions,
    cv_folds=cv_folds,
    seed=2222,
    generate_cache=False,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.classification
)

In [83]:
from torch.utils.data import random_split
from icu_benchmarks.data.loader import *
from torch.utils.data import DataLoader
bz = 256 
seed = 42 
split = False # Change depending on what data to use

if split:
    # Load test set 
    test_dataset = BATPolarsDataset(data=data, split="test", ram_cache=False, runmode=RunMode.classification)
    #test_loader = DataLoader(test_dataset, batch_size=128, collate_fn=test_dataset.collate_fn_pad_to_longest_in_batch()) # maybe change bz chgange log of that run 
    print(f'Test dataset length: {len(test_dataset)}')

    # Split the test set for fine-tuening 
    dataset_len = len(test_dataset)
    finetune_len = dataset_len // 2
    eval_len = dataset_len - finetune_len

    finetune_set, eval_set = random_split(test_dataset, [finetune_len, eval_len])

    finetune_loader = DataLoader(finetune_set, batch_size=bz, shuffle=True,
                                collate_fn=test_dataset.collate_fn_pad_to_longest_in_batch())
    eval_loader = DataLoader(eval_set, batch_size=bz, shuffle=False,
                            collate_fn=test_dataset.collate_fn_pad_to_longest_in_batch())

    print(f'Finetuening dataset length: {len(finetune_set)}')
    print(f'Evaluation dataset length: {len(eval_set)}')

else:
    finetune_train_set = BATPolarsDataset(data=data, split="train", ram_cache=False, runmode=RunMode.classification)
    finetune_val_set = BATPolarsDataset(data=data, split="val", ram_cache=False, runmode=RunMode.classification)
    finetune_test_set = BATPolarsDataset(data=data, split="test", ram_cache=False, runmode=RunMode.classification)

    finetune_train_loader = DataLoader(finetune_train_set, batch_size=bz, shuffle=True,
                                 collate_fn=finetune_train_set.collate_fn_pad_to_longest_in_batch())
    finetune_val_loader = DataLoader(finetune_val_set, batch_size=bz, shuffle=True,
                                 collate_fn=finetune_val_set.collate_fn_pad_to_longest_in_batch())
    eval_loader = DataLoader(finetune_test_set, batch_size=bz, shuffle=False,
                            collate_fn=finetune_test_set.collate_fn_pad_to_longest_in_batch())

    print(f'Finetuening training dataset length: {len(finetune_train_set)}')
    print(f'Finetuening validation dataset length: {len(finetune_val_set)}')
    print(f'Finetuening test dataset length: {len(finetune_test_set)}')

Finetuening training dataset length: 9506
Finetuening validation dataset length: 2377
Finetuening test dataset length: 2971


#### Testing

In [84]:
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm

# Ensure model is in evaluation mode
model.eval()
model.to("cpu")  # or "cuda" if using GPU

loss_fn = torch.nn.CrossEntropyLoss()

total_test_loss = 0
all_test_labels = []
all_test_probs = []

with torch.no_grad():
    test_loop = tqdm(eval_loader, desc="🧪 Evaluating on Test Set")

    for batch in test_loop:
        # You may need to adapt this unpacking depending on your dataset class
        x, mask, label, time, static, *_ = batch

        x = x.to("cpu").float()
        mask = mask.to("cpu").float()
        time = time.to("cpu").float()
        static = static.to("cpu").float()
        label = label.to("cpu").long()

        # Forward pass through BAT
        logits = model(x, static=static, time=time, sensor_mask=mask)

        loss = loss_fn(logits, label)
        total_test_loss += loss.item()

        # Collect outputs
        probs = F.softmax(logits, dim=1)[:, 1]  # Prob of class 1
        all_test_labels.extend(label.cpu().numpy())
        all_test_probs.extend(probs.cpu().numpy())

        test_loop.set_postfix(loss=loss.item())

# Final metrics
avg_test_loss = total_test_loss / len(eval_loader)
test_auroc = roc_auc_score(all_test_labels, all_test_probs)
test_auprc = average_precision_score(all_test_labels, all_test_probs)

print(f"\n🎯 Test Set Results:")
print(f"   Loss : {avg_test_loss:.4f}")
print(f"   AUROC: {test_auroc:.4f}")
print(f"   AUPRC: {test_auprc:.4f}")


🧪 Evaluating on Test Set:   0%|                                                                                                                                      | 0/12 [00:00<?, ?it/s]

🧪 Evaluating on Test Set: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [01:15<00:00,  6.33s/it, loss=0.32]


🎯 Test Set Results:
   Loss : 0.2911
   AUROC: 0.8086
   AUPRC: 0.3871


In [77]:
from pytorch_lightning import Trainer
from icu_benchmarks.models.wrappers import *

# Load your model from checkpoint
model = model.load_from_checkpoint(model_path, map_location="cpu")
model.eval()

# Your test_loader (already defined)
trainer = Trainer(accelerator="cpu", devices=1)

# This will compute loss, AUROC, AUPRC using model.test_step
test_metrics = trainer.test(model, dataloaders=eval_loader)
print(test_metrics)


TypeError: The classmethod `BAT.load_from_checkpoint` cannot be called on an instance. Please call it on the class type and make sure the return value is used.

In [68]:
import json

test_metrics_path = "/work3/s185395/yaib_logs/mimic/Mortality24/BAT/2025-05-31T11-18-01/repetition_0/fold_0/test_metrics.json"

with open(test_metrics_path, "r") as f:
    test_metrics = json.load(f)

print(test_metrics)


{'loss': 0.2890319228172302, 'AUC': 0.8105179071426392, 'PR': 0.43007785081863403}
